# Figures for the revised SoK (submission #401)

Regenerates `figures/fig_taxonomy.pdf`, `figures/fig_benchmarks.pdf`, and `figures/fig_mitigation.pdf`
just under the ACM sigconf column width (3.30 in versus 3.32 in) with every text element at 8 pt or larger, so the figures
stay readable when placed with `width=\columnwidth`. Every number plotted in Figure 2 is listed
with its source in the `RATES` table below; nothing is estimated or read off another figure.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from pathlib import Path

OUT = Path("figures"); OUT.mkdir(exist_ok=True)
COL_W = 3.30          # just under the ACM sigconf column (240 pt = 3.32 in), so scaling to \columnwidth enlarges text slightly
FS = 8                # minimum font size in points
plt.rcParams.update({
    "font.family": "Liberation Sans", "font.size": FS, "axes.titlesize": FS, "axes.labelsize": FS,
    "xtick.labelsize": FS, "ytick.labelsize": FS, "legend.fontsize": FS,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

def box(ax, x, y, w, h, text, fc, ec="#1f2937", tc="black", bold=False, fs=FS):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.004,rounding_size=0.012",
                                fc=fc, ec=ec, lw=0.6))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", color=tc, fontsize=fs,
            fontweight="bold" if bold else "normal", linespacing=1.05)

def elbow(ax, x0, y0, x1, y1):
    xm = (x0 + x1) / 2
    ax.plot([x0, xm, xm, x1], [y0, y0, y1, y1], color="#4b5563", lw=0.6, solid_capstyle="butt")

## Figure 1: taxonomy (after Ye et al., two axes)

Left-to-right tree. Progressive and regressive capitulation are marked because they are defined only
for instruments that judge responses against a correctness label (reference type R).

In [2]:
H1 = 1.95
fig = plt.figure(figsize=(COL_W, H1)); ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, COL_W); ax.set_ylim(0, H1); ax.axis("off")
DARK, MID, LIGHT = "#1e3a5f", "#2f6f9f", "#cfe2f3"
leaves = [("Progressive*", "prop"), ("Regressive*", "prop"),
          ("Flattery, unearned praise", "soc"), ("Moral-conflict affirmation", "soc"),
          ("Agreement with false claim", "overt"),
          ("Selective framing, omission", "covert"), ("Deferential hedging", "covert")]
top, bot = H1 - 0.06, 0.30
step = (top - bot) / (len(leaves) - 1)
LX = 1.80                      # x where leaf text starts
ly = {}
for i, (t, g) in enumerate(leaves):
    y = top - i * step
    ax.text(LX + 0.05, y, t, ha="left", va="center", fontsize=FS)
    ly.setdefault(g, []).append(y)
CW, CX = 0.74, 0.98            # class boxes
cy = {}
for t, g in [("Propositional", "prop"), ("Social", "soc"), ("Overt", "overt"), ("Covert", "covert")]:
    yc = sum(ly[g]) / len(ly[g]); cy[g] = yc
    box(ax, CX, yc - 0.11, CW, 0.22, t, LIGHT)
    for y in ly[g]:
        elbow(ax, CX + CW, yc, LX + 0.02, y)
AW, AX = 0.62, 0.30            # axis boxes
ay = {}
for t, gs in [("Target", ["prop", "soc"]), ("Directness", ["overt", "covert"])]:
    ya = (cy[gs[0]] + cy[gs[1]]) / 2; ay[t] = ya
    box(ax, AX, ya - 0.12, AW, 0.24, t, MID, tc="white", bold=True)
    for g in gs:
        elbow(ax, AX + AW, ya, CX, cy[g])
yr = (ay["Target"] + ay["Directness"]) / 2
ax.text(0.10, yr, "Sycophancy", ha="center", va="center", fontsize=FS, fontweight="bold", color=DARK, rotation=90)
for t in ay:
    elbow(ax, 0.20, yr, AX, ay[t])
ax.text(0.02, 0.06, "* Outcome classes defined only under correctness references (R).",
        ha="left", va="bottom", fontsize=FS, style="italic", color="#374151")
fig.savefig(OUT / "fig_taxonomy.pdf"); fig.savefig(OUT / "fig_taxonomy.png", dpi=200)
plt.close(fig)

## Figure 2: reported rates, split by reference type

Each panel uses its own metric; values are not comparable across panels, and rows within a panel
still differ in elicitation. A bar spans the lowest to the highest model the source reports; a dot is a
single aggregate or a single named model, as labeled. Sources (bib keys in brackets):

| Row | Value | Source |
|---|---|---|
| SycEval | 43.52% progressive + 14.66% regressive (58.19% overall), 3 assistants | [fanous2025syceval] |
| PARROT | follow rate 4% (GPT-5) to 94% (Qwen 2.5-1.5B), 22 models | [celebi2025parrot], abstract |
| FlipFlop | flips 46% on average, 10 LLMs | [laban2023flipflop], abstract |
| BrokenMath | best model (GPT-5) sycophantic 29%, 10 LLMs | [petrov2025brokenmath], abstract |
| CAP | stance reversal 5% to 56%, 17 LLMs | [jiang2026cap], abstract |
| llm-bias-bench | median 50.0% (direct) vs 78.9% (debate), 13 assistants | [nogueira2026llmbias], Sec. 4.3 |
| SycoEval-EM | acquiescence 0% to 100%, 19 LLMs | [peng2026sycoeval], abstract |
| MedPRESS | 86.8% of five-turn conversations with an unsafe agreement, 20 LLMs | [joy2026medpress], Sec. 6 |
| ELEPHANT | affirms either side of a moral conflict in 48% of cases, 11 LLMs | [cheng2026elephant] |

Evidence-calibrated (E) instruments report Bayesian error or switch curves rather than a rate and are
omitted.

In [3]:
PANELS = [
  ("(a) Correctness label (R): capitulation, % of items", [
      ("SycEval (3)",     "stack", (43.52, 14.66), "58.19 total"),
      ("PARROT (22)",     "range", (4, 94),        "4 to 94"),
      ("FlipFlop (10)",   "dot",   46,             "mean 46"),
      ("BrokenMath (10)", "dot",   29,             "best 29"),
  ]),
  ("(b) Counterfactual (C): reversal toward preference, %", [
      ("CAP (17)",            "range", (5, 56),      "5 to 56"),
      ("llm-bias-bench (13)", "pair",  (50.0, 78.9), "50 to 79"),
  ]),
  ("(c) Human or clinical norm (N): violations, % of cases", [
      ("SycoEval-EM (19)", "range", (0, 100), "0 to 100"),
      ("MedPRESS (20)",    "dot",   86.8,     "86.8"),
      ("ELEPHANT (11)",    "dot",   48,       "48"),
  ]),
]
C_RANGE, C_DOT, C_PROG, C_REGR = "#2f6f9f", "#1e3a5f", "#a9cbe6", "#1e3a5f"
ROW = 0.165; HEAD = 0.20; TICKS = 0.20
H2 = sum(len(p[1]) * ROW + HEAD for p in PANELS) + TICKS + 0.04
fig = plt.figure(figsize=(COL_W, H2))
L, R_ = 1.02 / COL_W, 1 - 0.50 / COL_W        # axes span in figure fraction
y_top = 1.0
for k, (title, rows) in enumerate(PANELS):
    n = len(rows)
    h = n * ROW / H2
    y_top -= HEAD / H2
    ax = fig.add_axes([L, y_top - h, R_ - L, h])
    y_top -= h
    for i, (lab, kind, v, note) in enumerate(rows):
        y = n - 1 - i
        if kind == "stack":
            ax.barh(y, v[0], height=0.55, color=C_PROG, edgecolor=C_DOT, lw=0.5)
            ax.barh(y, v[1], left=v[0], height=0.55, color=C_REGR, edgecolor=C_DOT, lw=0.5)
            ax.text(v[0] / 2, y, "progressive", ha="center", va="center", fontsize=FS, color="#0b1f33")
            ax.text(v[0] + v[1] + 1.5, y, "regressive", ha="left", va="center", fontsize=FS)
        elif kind == "range":
            ax.plot(v, [y, y], color=C_RANGE, lw=3.0, solid_capstyle="butt")
        elif kind == "pair":
            ax.annotate("", xy=(v[1], y), xytext=(v[0], y),
                        arrowprops=dict(arrowstyle="-|>", color=C_RANGE, lw=1.1, mutation_scale=7))
            ax.plot([v[0]], [y], "o", color=C_DOT, ms=3.2)
        else:
            ax.plot([v], [y], "o", color=C_DOT, ms=3.8)
        ax.text(103, y, note, ha="left", va="center", fontsize=FS, clip_on=False)
    ax.set_yticks(range(n)); ax.set_yticklabels([r[0] for r in rows][::-1])
    ax.set_ylim(-0.6, n - 0.4); ax.set_xlim(0, 100)
    ax.set_xticks([0, 25, 50, 75, 100])
    last = k == len(PANELS) - 1
    ax.tick_params(axis="x", labelbottom=last, length=2 if last else 0, pad=1)
    ax.tick_params(axis="y", length=0, pad=2)
    ax.text(-1.02 * 100 / ((R_ - L) * COL_W) * 1.0 + 0, n - 0.4 + 0.08, title, transform=ax.transData,
            ha="left", va="bottom", fontsize=FS, fontweight="bold")
    ax.grid(axis="x", color="#d1d5db", lw=0.5); ax.set_axisbelow(True)
    for s_ in ("top", "right", "left"): ax.spines[s_].set_visible(False)
fig.savefig(OUT / "fig_benchmarks.pdf"); fig.savefig(OUT / "fig_benchmarks.png", dpi=200)
plt.close(fig)

## Figure 3: four control tiers with evidence grades

Examples per tier with their Table 6 grade (A: at least two model families, at least two domains, and
multi-turn or held-out testing; B: two of these; C: one; D: none). Policy-tier entries are evaluation
practices and are not graded.

In [4]:
import textwrap
TIERS = [
  ("Data", "#dbeafe", "#1e3a5f", "Synthetic anti-sycophancy data (B)"),
  ("Training", "#dcfce7", "#166534", "Pinpoint tuning (A), CAUSM (A), SMART (A), Lag-DPO (A), reward decomposition (A), consistency training (B), Holistic DPO (B)"),
  ("Inference", "#ede9fe", "#5b21b6", "Third-person prompt (A), activation scaling (A), peer priors (A), MONICA (B), CAA (C), Ask Don't Tell (C)"),
  ("Policy", "#fef3c7", "#92400e", "Pander Score (AEDI) tracking; multi-turn robustness testing (not graded)"),
]
LINE = FS * 1.25 / 72          # line height in inches
wrapped = [textwrap.fill(t[3], 46) for t in TIERS]
heights = [max(0.24, (w.count("\n") + 1) * LINE + 0.08) for w in wrapped]
GAP = 0.09
H3 = sum(heights) + GAP * (len(TIERS) - 1) + 0.02
fig = plt.figure(figsize=(COL_W, H3)); ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, COL_W); ax.set_ylim(0, H3); ax.axis("off")
y = H3 - 0.01; spans = []
for (lab, fc, hc, _), w, h in zip(TIERS, wrapped, heights):
    y0 = y - h
    box(ax, 0.01, y0, 0.66, h, lab, hc, tc="white", bold=True)
    box(ax, 0.75, y0, COL_W - 0.76, h, w, fc)
    spans.append((y0, y)); y = y0 - GAP
for (b0, _), (_, t1) in zip(spans[:-1], spans[1:]):
    ax.annotate("", xy=(0.34, t1), xytext=(0.34, b0),
                arrowprops=dict(arrowstyle="-|>", color="#4b5563", lw=0.7, mutation_scale=6))
fig.savefig(OUT / "fig_mitigation.pdf"); fig.savefig(OUT / "fig_mitigation.png", dpi=200)
plt.close(fig)

## Check: every text element is at least 8 pt

Prints each PDF's page size: every figure is 3.30 in (237.6 pt) wide, just under the 240 pt column, so `width=\columnwidth` enlarges it by about 1% and no text falls below 8 pt.

In [5]:
import subprocess, json
for f in ["fig_taxonomy", "fig_benchmarks", "fig_mitigation"]:
    info = subprocess.run(["pdfinfo", str(OUT / f"{f}.pdf")], capture_output=True, text=True).stdout
    size = [l for l in info.splitlines() if l.startswith("Page size")][0]
    print(f, size)

fig_taxonomy Page size:       237.6 x 140.4 pts
fig_benchmarks Page size:       237.6 x 167.4 pts
fig_mitigation Page size:       237.6 x 135.44 pts
